# Essay Scoring Data Preparation

Private research run. Source snapshot: `5b76ac7ec39d5bd856cee2ce2dce2d1542a66e77f414df291f768464a67ac44a`.
ELLIPSE and IELTS labels are separate. Website labels do not establish examiner agreement.


In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
project = pathlib.Path("/kaggle/temp/essay-scoring-code")
project.mkdir(parents=True, exist_ok=True)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '5b76ac7ec39d5bd856cee2ce2dce2d1542a66e77f414df291f768464a67ac44a'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(project)
os.environ["PYTHONPATH"] = str(project / "src")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/kaggle/temp/huggingface"
os.environ["WANDB_DISABLED"] = "true"
os.chdir(project)
print("Source bundle SHA256:", '5b76ac7ec39d5bd856cee2ce2dce2d1542a66e77f414df291f768464a67ac44a')
pathlib.Path("/kaggle/working/source_bundle_sha256.txt").write_text('5b76ac7ec39d5bd856cee2ce2dce2d1542a66e77f414df291f768464a67ac44a')


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyzipper==0.3.6"], check=True)
subprocess.run([sys.executable, "-m", "ielts_scoring.data", "--output", "/kaggle/temp/essay-data"], check=True)
import shutil, importlib.metadata
for track_name in ["ellipse", "ielts"]:
    source = pathlib.Path("/kaggle/temp/essay-data/prepared") / track_name
    shutil.copytree(source, pathlib.Path("/kaggle/working") / track_name, dirs_exist_ok=True)
packages = {n: importlib.metadata.version(n) for n in ["pandas", "numpy", "scikit-learn", "pyarrow", "requests", "pyzipper"]}
pathlib.Path("/kaggle/working/preparation_environment.json").write_text(json.dumps(packages, indent=2))
print("Prepared datasets saved. No GPU was used.")
